# 创建 SOLT 校准标准

## 简介在 scikit-rf 中，校准标准被视为一个普通的单端口或双端口 `skrf.Network`，其定义方式是基于其完整的 S 参数。它可以代表反射、传输、负载，甚至任意阻抗的标准。由于在电路建模和拟合中没有引入额外的误差，因此这种方法可以实现最高的校准精度，并且在 VNA 供应商的术语中被称为“基于数据的标准”。但是，VNA 校准标准通常使用具有拟合系数的电路模型来定义。由于基于数据的标准通常作为优质产品提供，因此许多基于模型的校准套件仍在被使用和制造（甚至在 50 GHz 时 [[1](SOLT%20Calibration%20Standards%20Creation.html#ref1)]）。这需要在它们被用于 scikit-rf 的校准例程之前创建它们的网络模型。本示例解释了如何从 HP-Agilent-Keysight 格式和 Rohde & Schwarz / Anritsu 格式中给出的系数创建网络。两者本质上是相同的电路模型，但后者格式对偏移传输线使用不同的测量单位。作为示例，本文还绘制了 Keysight 85032F（Type-N 插件）、Keysight 85033E（3.5 mm 插件）、通用齐平标准（SMA 插件）和 Maury Microwave 8050CK10（3.5 mm 插件）的计算响应。.. 注意：本指南仅涵盖同轴标准。对于波导，计算方法不同。特别是，对于同轴线，乘以 $\sqrt{\text{GHz}}$ 的标定不能应用于波导，因为损耗也是其物理尺寸的函数，具有更复杂的公式。您是否有波导方面的经验？如果有，您可以帮助[为文档做出贡献](../../contributing/index.rst#examples-and-tutorials)。## scikit-rf 建模的替代方案在我们开始之前，值得指出一些替代方案。在 scikit-rf 中，您可以利用其 S 参数使用任何现有的标准定义。如果您已经将标准定义为其他工具（例如，您最喜欢的电路仿真器或实际测量结果）中的网络，则可以简单地将 S 参数导出到 Touchstone 文件中，以便在 scikit-rf 中使用。同样，如果您已经在使用基于数据的校准标准，则应该可以直接使用其数据。S 参数可以存储在特定于设备的的文件格式中，请咨询您的供应商，了解是否可以将其导出为 Touchstone 文件。作为一种特殊情况，如果使用“齐平”（零长度）标准——这意味着标准直接位于连接器的参考平面上，没有延伸的部件——有时可以假定校准标准对于非关键测量是理想的。在 scikit-rf 中，可以通过定义一个理想的传输线并调用 `short()`、`open()`、`match()` 和 `thru()` 方法（在[准备](#preparation)部分中解释）来方便地创建理想的响应。.. 重要提示：理想的假设仅对“齐平”标准在近似上有效。大多数实验室级标准都是“偏移”标准，请始终使用正确的定义来使用这些标准。它们具有来自其延伸部件的额外电延迟，在理想假设下，相位误差将是不可接受的。将“齐平”标准放置在“直通”适配器之后会产生相同的相位偏差。请参见下文，了解“齐平”标准和“偏移”标准之间的区别。## HP-Agilent-Keysight 系数格式在介绍必要的背景知识之后，让我们开始吧。为了本指南的目的，我们将对 Keysight 85032F、Type-N、50 Ω、DC 到 9 GHz 校准套件（插件）进行建模，其系数如下。| 参数 | 单位 | 开路 | 短路 | 负载 | 直通 ||---|---|---|---|---|---|| $\text{C}_0$ | $10^{-15} \text{ F}$ | 89.939 | | | || $\text{C}_1$ | $10^{-27} \text{ F/Hz}$ | 2536.800 | | | || $\text{C}_2$ | $10^{-36} \text{ F/Hz}^2$ | -264.990 | | | || $\text{C}_3$ | $10^{-45} \text{ F/Hz}^3$ | 13.400 | | | || $\text{L}_0$ | $10^{-12} \text{ H}$ | | 3.3998 | | || $\text{L}_1$ | $10^{-24} \text{ H/Hz}$ | | -496.4808 | | || $\text{L}_2$ | $10^{-33} \text{ H/Hz}^2$ | | 34.8314 | | || $\text{L}_3$ | $10^{-42} \text{ H/Hz}^3$ | | -0.7847 | | || 阻抗 | $\Omega$ | | | 50 | || 偏移延迟 | ps | 40.856 | 45.955 | 0 | 0 || 偏移损耗 | $\text{G}\Omega$ / s | 0.93 | 1.087 | 0 | 0 || 偏移 $Z_0$ | $\Omega$ | 50 | 49.992 | 50 | 50 || 参考 $Z_0$ | $\Omega$ | 50 | 50 | 50 | 50 |

### 电路模型在开始创建网络定义之前，我们需要先了解底层的电路模型以及这些系数的含义。如图所示，这是惠普-安捷伦-科仪（HP-Agilent-Keysight）公司校准标准的模型。<figure><img src="solt_calibration_standards_creation/calkit-schematic.svg" width="40%"><figcaption> *校准标准的电路模型* </figcaption></figure>#### 终端阻抗在*偏移*传输线的末端连接了一个分路阻抗，用于模拟开路或短路标准中的分布式电容或电感。它表示为一个三阶多项式，具有四个系数，$y(f) = a_0 + a_1 f + a_2 f^2 + a_3 f^3$，其中 $f$ 是频率，$a_i$ 是系数。对于开路标准，它们是 $\text{C}_0$、$\text{C}_1$、$\text{C}_2$、$\text{C}_3$，第一个常数项的单位是飞法（femtofarad）。对于短路标准，它们是 $\text{L}_0$、$\text{L}_1$、$\text{L}_2$、$\text{L}_3$，第一个常数项的单位是皮亨（picohenry）。.. important::  终端阻抗的反射系数 ($S_{11}$) 是相对于系统阻抗 ($Z_\mathrm{ref}$) 计算的，而不是相对于偏移传输线的阻抗 ($Z_\text{off}$ 或 $Z_\text{c}$) 计算的。 [[15](SOLT%20Calibration%20Standards%20Creation.html#ref15)]#### 偏移线在校准标准的前面，存在一条*偏移*有损传输线。为了理解其含义，我们需要区分两种类型的校准标准：*紧贴*（零长度）标准和*偏移*标准。在*紧贴*标准中，物理标准直接位于端口的参考平面上。这种设计广泛用于未表征的通用 SMA 校准“标准”（如果可以这样称呼的话），例如业余无线电中。*紧贴*标准并不总是质量差，它们也存在于一些实验室级别的标准中，例如用于 APC-7 连接器的科仪 85031B。

<figure><img src="solt_calibration_standards_creation/flush-calkit.jpg" width="45%"><img src="solt_calibration_standards_creation/offset-calkit.jpg" width="45%"><figcaption>*紧密贴合的校准标准紧贴连接器，没有任何额外的结构。偏移校准标准具有延伸的本体，引入了电延迟。*</figcaption></figure>然而，大多数实验室级校准标准被设计为*偏移*校准标准，连接器参考平面和标准物理位置之间存在一个较短的长度。这样做是为了提高标准电气特性的可预测性 [[2](SOLT%20Calibration%20Standards%20Creation.html#ref2)]，或者仅仅是因为连接器的机械设计存在局限性（例如，所有Type-N连接器都具有5.258-5.360 mm的参考平面偏移量[[3](SOLT%20Calibration%20Standards%20Creation.html#ref3)]）。这两种因素都会引入一个必须考虑的电长度，否则会导致很大的相位偏差。这种*偏移*长度被建模为一个传输线，使用三个参数：1. **无损偏移阻抗** ($Z_\text{off}$)：一个实数特性阻抗，忽略了线路损耗（$R$和$G$）。   这通常与VNA的参考阻抗匹配，但并不总是如此。有时会使用与参考阻抗略有不同的值来模拟物理缺陷，例如50.209 Ω或49.992 Ω。   此外，波导标准使用一个特殊的归一化值`1`。2. **偏移延迟** ($t_\text{delay}$)：单向电延迟，以皮秒（ps）为单位。3. **偏移损耗** ($A_\text{loss}$)。在1 GHz时，单位时间内（以秒为单位）的标称衰减，以GΩ/s（十亿欧姆/秒）为单位。将此值按$\sqrt{f / (\text{1 GHz})}$的比例缩放，以用于其他频率。   GΩ/s单位虽然不常见，但有其原因。要表示传输线段的长度，可以使用距离（以米为单位）或时间延迟（以秒为单位）。相对于时间定义线路，可以计算出$R = A_\text{loss} t_\text{delay}$，单位为欧姆，而无需显式定义介电常数$\epsilon$。.. important::    这些参数在数据表中以缩放单位给出，但在所有计算中应使用未缩放的SI单位（无前缀）：欧姆（Ω）、秒（s）、欧姆/秒（Ω/s）和赫兹（Hz）。##### RLCG参数<figure><img src="solt_calibration_standards_creation/rlcg.svg" width="40%"><figcaption>*RLCG传输线模型*</figcaption></figure>由于这些参数不是标准的，因此我们需要将它们转换为熟悉的RLCG线路参数。对[[4](SOLT%20Calibration%20Standards%20Creation.html#ref4)]，公式2B.4进行代数运算后，可以计算出相应的与频率相关的RLCG参数，如下所示。$$\begin{aligned}R(f) \cdot l &= \left(A_\text{loss} t_\text{delay}\right) \cdot \sqrt{\frac{f}{10^9}} \\L(f) &= L_0 + L_\text{cond} = t_\text{delay}  Z_\text{off} + \dfrac{R}{2\pi f} \\C &= \dfrac{t_\text{delay}}{Z_\text{off}} \\G &= 0 \\l &= 1\text{ (归一化)}\end{aligned}$$其中，$L_0$是理想线路的电感，$L_\text{cond} = R / (2 \pi f)$是不良导体由于趋肤效应而产生的与频率相关的电感。在[[5](SOLT%20Calibration%20Standards%20Creation.html#ref5)]中，推导出了几乎相同的公式，只是省略了$L_\text{cond}$这一项。为了与原始Keysight的出版物[[4](SOLT%20Calibration%20Standards%20Creation.html#ref4)]保持一致，必须将其包含在内（否则，在9 GHz时，85032F的相位误差将大于0.1度）。使用这些参数，可以计算出有损线路的*传播常数*（$\gamma$）和复数*特性阻抗*（$Z_c$）。请注意，在与频率相关的`RLCG(f)`模型中，每个频率都有其自身的RLCG参数和线路常数。.. important:: 我们的公式*不完全*与官方Keysight的计算和其他工作相同。后者在计算线路常数时使用了低损耗近似，而我们的公式是从RLCG参数开始，推导出精确的线路常数。两种公式的差异都非常小（Keysight 85056A的幅度和相位角在50 GHz时，精度达到小数点后4位），这种差异只是学术上的。.. note:: 我们展示了RLCG方法，因为它被广泛理解且易于使用。如果绝对需要再现到最后一位数字的精确数值，请参阅*附录*，其中包含Keysight官方低损耗近似的推导和代码。#### 参考阻抗所有校准标准都隐含了一个*参考阻抗* $Z_\text{ref}$。一些Keysight的数据表将其显示为“系统$Z_0$”[[6](SOLT%20Calibration%20Standards%20Creation.html#ref6)]，但并不总是明确给出。可能需要一些上下文才能推断出它：如果一个标准命名为*Keysight 85032F Type N (50) Calibration Kit*，那么$Z_\text{ref}$就是50 Ω。不要将其与偏移传输线的特性阻抗$Z_0$（无损）或$Z_c$（有损）混淆，后者的值可以是49.992 Ω。与*线路阻抗*不同，*参考阻抗*的目的是将S参数重新归一化为标准形式，因为所有S参数都是相对的。它始终设置为VNA系统的参考阻抗：50 Ω或75 Ω。##### 连接器不连续性由于参考阻抗似乎是一个任意选择，人们可能会得出结论，任何校准标准都可以用于校准任何VNA端口（只要它们在机械上兼容），而与偏移线路阻抗无关。例如，人们可能会尝试通过将其直接连接到75 Ω校准标准来校准50 Ω VNA端口，以便所有校准后的测量结果都表现得好像是使用75 Ω系统进行的。同样，如果对75 Ω校准标准的参考阻抗进行重新归一化，似乎可以预测其在50 Ω VNA上的理想S参数，从而将其用作50 Ω标准。但是，这个假设是不正确的。重新归一化无法预测由于传输线物理尺寸的突然变化而产生的物理测试端口的寄生效应。[[7](SOLT%20Calibration%20Standards%20Creation.html#ref7)][[9](SOLT%20Calibration%20Standards%20Creation.html#ref9)]它近似为一个电容不连续性。[[18](SOLT%20Calibration%20Standards%20Creation.html#ref18)]即使两端具有相同的特性阻抗，这种效应也存在，只要同轴线的物理尺寸发生阶跃变化（例如，从3.5 mm到2.92 mm端口）。[[8](SOLT%20Calibration%20Standards%20Creation.html#ref8)]从理论上讲，不连续处的高阶模式的激发会使校准不可靠，因为它可以对扰动敏感且不可重复。事实上，在50 GHz时，甚至在两个“完美”1.85 mm连接器之间也观察到病态情况——配对的连接器由于其机械公差*太精确*而产生共振，近乎零的间隙形成了一个谐振器。[[25](SOLT%20Calibration%20Standards%20Creation.html#ref25)][[26](SOLT%20Calibration%20Standards%20Creation.html#ref26)]然而，在大多数不受此类病态现象影响的实际测量中，已经表明，如果DUT和校准标准具有相同的连接器设计，它们都会产生类似的不连续性，因此这种不连续性通常可以被校准掉，从而只留下一个微小的误差，对于所有最严格的计量应用来说，该误差都可以忽略不计。[[24](SOLT%20Calibration%20Standards%20Creation.html#ref24)] 这就是使用50 Ω端口校准后，测量75 Ω器件的情况。[[9](SOLT%20Calibration%20Standards%20Creation.html#ref9)] 但是，如果DUT和校准标准具有不同的连接器设计（例如，交叉连接3.5 mm、2.92 mm和SMA），则这种不连续性会产生轻微的测量误差。[[8](SOLT%20Calibration%20Standards%20Creation.html#ref8)]因此，为了获得令人满意的结果，校准标准和DUT必须使用相同的连接器。为了获得最佳结果，测试端口（或适配器）必须具有高质量，并且与校准标准匹配。#### 拟合的非唯一性和物理可解释性小电容或电感的相位变化大约是线性的，因此通常可以用作终止电抗或偏移传输线中的电延迟来对其进行建模。[[4](SOLT%20Calibration%20Standards%20Creation.html#ref4)][[10](SOLT%20Calibration%20Standards%20Creation.html#ref10)]例如，Keysight 85033D的数据表显示了短路标准的两种模型，一种是纯时间延迟，另一种是带有电感的延迟。[[6](SOLT%20Calibration%20Standards%20Creation.html#ref6)]由于标准定义的非唯一性，在解释标准定义中的各个术语时需要谨慎：整体响应是有意义的，各个术语可能是有意义的，也可能没有意义。大多数实验室级校准标准试图确保标准定义在物理上是有意义的[[11](SOLT%20Calibration%20Standards%20Creation.html#ref11)]，但这并不总是得到保证。在通过曲线拟合定义自定义校准标准时，很难区分由于偏移传输线和终止电抗引起的相位变化。在先验知识下，可以通过机械测量固定偏移长度[[12](SOLT%20Calibration%20Standards%20Creation.html#ref12)]，或通过电磁理论或仿真固定终止电容[[10](SOLT%20Calibration%20Standards%20Creation.html#ref10)]。这两种方法都需要对标准的精确知识和建模。例如，历史研究侧重于空心充气开放标准（使用截止频率以下的圆形波导理论）[[10](SOLT%20Calibration%20Standards%20Creation.html#ref10)]，但由于许多开放标准使用介电支撑，因此它不再适用。在缺乏信息的情况下，仅延迟模型可能是唯一的实际选择。.. tip::   事实上，仅延迟的短路标准是较老的VNA固件所支持的唯一模型，例如HP 8510[[13](SOLT%20Calibration%20Standards%20Creation.html#ref13)]。   通过scikit-rf进行外部校准可以让你绕过此限制。#### 忽略的术语无论是开路还是短路，都可以将其建模为仅具有电延迟，而没有电抗，反之亦然。仅延迟模型是最常用的短路标准，因为它在历史上一直被使用。匹配负载产生的反射非常小，因此通常将其简单地建模为$Z_0$终端。由于其反射系数约为0，因此其相位角没有意义，因此其电延迟也被忽略。如果认为损耗可以忽略不计，则直通标准有时会建模为仅具有偏移延迟。一些校准套件没有直通延迟数据。这些套件用于*紧密贴合直通*、*交换等长适配器*或*未知直通*校准，而不使用*直通*的数据。*紧密贴合直通*是传统的SOLT校准方法：端口1和端口2直接连接，不使用适配器。这种*直通*在定义上是理想的，并且具有零长度，不需要进行建模。在*交换等长适配器*校准中，涉及一对延迟匹配的适配器，这使得*直通*的实际长度变得不明显。*未知直通*校准算法相对较新，不需要对直通进行表征。.. note::   有时，偏移损耗仍然出现在负载和直通的定义中，但具有零电长度。这些参数应该被忽略。零偏移延迟禁用了偏移传输线。#### 插头和插座最后，值得澄清一下Keysight的插头和插座符号。如果使用*插头*标准来校准*插座*，则数据表会将其表示为相对于自身为`-m-`，或相对于测试端口为`(f)`。如果使用*插座*标准来校准*插头*，则数据表会将其表示为相对于自身为`-f-`，或相对于测试端口为`(m)`。[[14](SOLT%20Calibration%20Standards%20Creation.html#ref14)]

### 准备<div id="preparation"></div>有了这个电路模型，我们就可以开始对校准标准进行建模。首先，我们需要导入一些库定义，并指定计算的频率范围。这里，我们使用 1 MHz 到 9 GHz，共 1001 个点。您可以根据需要进行调整。我们还定义了一个 `ideal_medium`，其端口阻抗为 $50 \space\Omega$，以便用于后续的一些计算。

In [ ]:
import numpy as np

import skrf
from skrf.media import DefinedGammaZ0, DistributedCircuit

# reference impedance of the S-parameters (not the offset line)
z0_ref = 50
freq = skrf.Frequency(1, 9000, 1001, "MHz")
ideal_medium = DefinedGammaZ0(frequency=freq, z0=z0_ref)

### 理想情况下的响应首先了解特殊情况是有帮助的：理想的校准标准可以通过调用 `ideal_medium` 中的 `open()`、`short()`、`match()` 和 `thru()` 方法轻松创建。前三个方法返回一个 1 端口网络。`thru()` 方法返回一个 2 端口网络。

In [ ]:
ideal_open  = ideal_medium.open()
ideal_short = ideal_medium.short()
ideal_load  = ideal_medium.match()
ideal_thru  = ideal_medium.thru()

### 偏移传输线的建模为了正确地对偏移传输线进行建模，应该使用偏移延迟、偏移损耗和偏移 $Z_0$ 来推导出有损线路的 RLCG 参数。在内部，`scikit-rf` 会自动推导出有损线路的*传播常数* ($\gamma$) 和复数*特性阻抗* ($Z_c$)。偏移线路参数与 RLCG 线路参数之间的关系已经在上面的讨论中给出了。让我们将这些公式转换为代码。.. important:   $\sqrt{\frac{f}{10^9}}$ 这个项用于将线路损耗从标称 1 GHz 值缩放到给定的频率，但这仅适用于同轴线路。

In [ ]:
def offset_rlcg(freq, offset_delay, offset_loss, offset_z0):
    r = offset_loss * offset_delay * np.sqrt(freq.f / 1e9)
    l = (offset_delay * offset_z0) + r / (2 * np.pi * freq.f)
    c = offset_delay / offset_z0
    g = 0
    return (r, l, c, g)

.. 提示：这里使用了 `numpy` 中的广播功能。`r` 和 `l` 都是与频率相关的量，因此它们是数组，而不是标量。与其显式地循环遍历每个频率并将它们添加到数组中，不如通过标量与 `numpy.array` 的乘法自动创建数组。我们将继续使用这种技术。定义了函数 `offset_rlcg()` 后，我们可以通过调用它来计算开路和短路校准标准对应的线路参数。

In [ ]:
rlcg_open = offset_rlcg(freq, 40.856e-12, 0.93e9, 50)
rlcg_short = offset_rlcg(freq, 45.955e-12, 1.087e9, 49.992)

现在，我们已经掌握了关于该移相线的必要信息。任务的另一半非常简单：在 scikit-rf 中，使用这些 RLCG 参数创建一个用于该传输线的两端口网络。scikit-rf 将自动帮助我们计算其传播常数 $\gamma l$ 和复数特性阻抗 $Z_c$。在 scikit-rf 中执行此任务非常简单：1. 首先，创建一个 `DistributedCircuit` 介质，并将四个数组（RLCG）作为输入。创建的 `DistributedCircuit` 代表一个物理介质。2. 在创建介质时，还需要指定 S 参数参考阻抗 $Z_\text{ref}$，例如 `z0_port=50`（或 `z0_port=75`）。   * 请勿将此参考阻抗 $Z_\text{ref}$ 与移相线阻抗 $Z_\text{off}$ 和 $Z_c$ 混淆。$Z_\text{ref}$（实数）始终是 VNA 的标称系统或端口阻抗，而线阻抗 $Z_\text{off}$（实数）和 $Z_c$（复数）可能由于物理缺陷和损耗而有所不同。3. 然后，通过调用介质的 `line()` 方法，可以创建一个实际长度为 1 米的线。

In [ ]:
medium_open = DistributedCircuit(frequency=freq,
    R=rlcg_open[0], L=rlcg_open[1], C=rlcg_open[2], G=rlcg_open[3],
    z0_port=z0_ref
)
line_open = medium_open.line(
    d=1, unit='m'
)

medium_short = DistributedCircuit(frequency=freq,
    R=rlcg_short[0], L=rlcg_short[1], C=rlcg_short[2], G=rlcg_short[3],
    z0_port=z0_ref
)
line_short = medium_short.line(
    d=1, unit='m'
)

### 建模并联阻抗接下来，我们需要对开路和短路校准标准中的并联阻抗进行建模。对于开路标准，它是一个电容。对于短路标准，它是一个电感。两者都建模为关于频率的三次多项式。在 `numpy` 中，可以使用 `np.poly1d([x3, x2, x1, x0])` 快速定义这样的函数。这是一个高阶函数，它接受一个按降序排列的系数列表，并返回一个可调用的多项式函数。在计算多项式后，我们可以生成频率相关的电容和电感。开路建模为一个串联的 `medium.capacitor()`，然后是一个理想的 `medium.short()`。短路建模为一个串联的 `medium.inductor()`，然后是一个理想的 `medium.short()`。根据 [[15](SOLT%20Calibration%20Standards%20Creation.html#ref15)]，电容和电感的 S 参数是相对于系统的参考阻抗定义的，而不是 *偏移* 传输线，因此我们使用 `ideal_medium` 以避免混淆。然而，`medium_open` 或 `medium_short` 也是可以接受的：它们也正确地使用 `skrf.Media()` 的端口阻抗作为参考阻抗。

In [ ]:
# use ideal_medium, not medium_open and medium_short to avoid
# reference impedance confusions.

capacitor_poly = np.poly1d([
      13.400 * 1e-45,
    -264.990 * 1e-36,
    2536.800 * 1e-27,
      89.939 * 1e-15
])
capacitor_list = capacitor_poly(freq.f)
shunt_open = ideal_medium.capacitor(capacitor_list) ** ideal_medium.short()

inductor_poly = np.poly1d([
      -0.7847 * 1e-42,
      34.8314 * 1e-33,
    -496.4808 * 1e-24,
       3.3998 * 1e-12
])
inductor_list = inductor_poly(freq.f)
shunt_short = ideal_medium.inductor(inductor_list) ** ideal_medium.short()

### 关于串联和并联阻抗的说明为了对并联阻抗（如用于开路校准标准的电容）进行建模，可以使用一个串联的 `medium.capacitor()`，其一端连接到 `medium.short()`。```# [端口] ---- [电容] --- [端口]#                               |#                               |#                            [短路]#                               |#                               |#                              GNDshunt_open = ideal_medium.capacitor(capacitor_list) ** ideal_medium.short()```或者，可以使用一个并联的 `medium.shunt_capacitor()`，其一端连接到 `medium.open()`。```# [端口] -----v----- [端口] ----- [开路]#             |#             |#     [并联电容]#             |#             |#           [GND]shunt_open = ideal_medium.shunt_capacitor(capacitor_list) ** ideal_medium.open()```两者是等效的。`medium.open()` 端接对于 `shunt_capacitor()` 来说非常重要：它创建一个具有一个连接到地线的电容的二端口网络——这个网络类似于示波器端接器，这是一个具有并联 50 Ω 电阻的 2 端口器件——因此，另一个“输出”端口需要是开路。否则，仅由 `shunt_capacitor()` 端接的线路会产生不正确的散射参数。

### 完成

最后，我们将这些模型组件连接在一起，并添加理想负载和直通的定义，这样就完成了我们的建模。

In [ ]:
open_std = line_open ** shunt_open
short_std = line_short ** shunt_short
load_std = ideal_medium.match()
thru_std = ideal_medium.thru()

现在，您可以将这些校准标准传递给 scikit-rf 的校准程序，或者使用 `write_touchstone()` 方法将其保存到磁盘中，以供将来使用。.. 注意：这里，我们生成的 `open_std`、`short_std` 和 `load_std` 都是单端口网络，但 scikit-rf 的大多数校准程序都需要使用双端口网络作为校准标准，因为它们用于双端口校准。您可以使用 `skrf.two_port_reflect()` 函数从两个单端口网络生成一个双端口网络。有关更多信息，请务必阅读文档中的 [SOLT 校准](./SOLT.ipynb) 示例。

### 绘图

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

skrf.stylely()

最后，让我们看一下我们的校准标准的幅度和相位变化。#### 开路

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(12, 5)

plt.suptitle("Keysight 85032F Plug Open (S11)")
open_std.plot_s_db(ax=ax[0], color='red', label="Magnitude")
open_std.plot_s_deg_unwrap(ax=ax[1], color='blue', label="Phase")

#### 短路

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(12, 5)

plt.suptitle("Keysight 85032F Plug Short (S11)")
short_std.plot_s_db(ax=ax[0], color='red', label="Magnitude")
short_std.plot_s_deg_unwrap(ax=ax[1], color='blue', label="Phase")

### 结论如图所示，校准标准的损耗极低，整个频率范围内约为 0.01 dB。同时，相位偏移是真正需要进行补偿的部分。在 1 GHz 时，相位偏移已经达到 25 度左右。.. important::   大多数实验室级别的校准标准都是偏移标准，因此较大的相位偏移是不可避免的。在使用这些校准标准时，务必输入正确的定义。

### 代码片段为了方便起见，您可以重复使用以下代码片段，从 Keysight 格式的系数中生成校准标准的网络。

In [ ]:
import numpy as np

import skrf
from skrf.media import DefinedGammaZ0, DistributedCircuit


def keysight_calkit_offset_line(freq, offset_delay, offset_loss, offset_z0, ref_z0):
    if offset_delay or offset_loss:
        r = offset_loss * offset_delay * np.sqrt(freq.f / 1e9)
        l = (offset_delay * offset_z0) + r / (2 * np.pi * freq.f)
        c = offset_delay / offset_z0
        g = 0

        medium = DistributedCircuit(frequency=freq,
            R=r, L=l, C=c, G=g,
            z0_port=ref_z0
        )
        offset_line = medium.line(d=1, unit='m')
        return medium, offset_line
    else:
        medium = DefinedGammaZ0(frequency=freq, z0=ref_z0)
        line = medium.line(d=0)
        return medium, line


def keysight_calkit_open(freq, offset_delay, offset_loss, c0, c1, c2, c3, offset_z0, ref_z0):
    # Capacitance is defined with respect to the system reference impedance ref_z0, not the
    # lossy line impedance. In scikit-rf, the return values of `shunt_capacitor()` and
    # `medium.open()` methods are (correctly) referenced to z0_port, which has been set to
    # ref_z0.
    medium, line = keysight_calkit_offset_line(freq, offset_delay, offset_loss, offset_z0, ref_z0)
    if c0 or c1 or c2 or c3:
        poly = np.poly1d([c3, c2, c1, c0])
        capacitance = medium.shunt_capacitor(poly(freq.f)) ** medium.open()
    else:
        capacitance = medium.open()
    return line ** capacitance


def keysight_calkit_short(freq, offset_delay, offset_loss, l0, l1, l2, l3, offset_z0, ref_z0):
    # Inductance is defined with respect to the system reference impedance ref_z0, not the
    # lossy line impedance. In scikit-rf, the return values of `shunt_inductance()` and
    # `medium.short()` methods are (correctly) referenced to z0_port, which has been set to
    # ref_z0.
    medium, line = keysight_calkit_offset_line(freq, offset_delay, offset_loss, offset_z0, ref_z0)
    if l0 or l1 or l2 or l3:
        poly = np.poly1d([l3, l2, l1, l0])
        inductance = medium.inductor(poly(freq.f)) ** medium.short()
    else:
        inductance = medium.short()
    return line ** inductance


def keysight_calkit_load(freq, offset_delay=0, offset_loss=0, offset_z0=50, ref_z0=50):
    medium, line = keysight_calkit_offset_line(freq, offset_delay, offset_loss, offset_z0, ref_z0)
    ideal_medium = DefinedGammaZ0(frequency=freq, z0=ref_z0)
    load = ideal_medium.match()
    return line ** load


def keysight_calkit_thru(freq, offset_delay=0, offset_loss=0, offset_z0=50, ref_z0=50):
    medium, line = keysight_calkit_offset_line(freq, offset_delay, offset_loss, offset_z0, ref_z0)
    thru = medium.thru()
    return line ** thru

### 示例：Keysight 85033E为了展示上述代码片段的另一个使用示例，我们将模拟 Keysight 85033E，3.5 mm，50 Ω，DC 到 9 GHz 校准套件（插头），其系数如下。[[22](SOLT%20Calibration%20Standards%20Creation.html#ref22)]这是实验室中最常用的 3.5 mm 校准标准之一。| 参数 | 单位 | 开路 | 短路 | 负载 | 直通 ||---|---|---|---|---|---|| $\text{C}_0$ | $10^{-15} \text{ F}$ | 49.433 |  |  |  || $\text{C}_1$ | $10^{-27} \text{ F/Hz}$ | -310.13 |  |  |  || $\text{C}_2$ | $10^{-36} \text{ F/Hz}^2$ | 23.168 |  |  |  || $\text{C}_3$ | $10^{-45} \text{ F/Hz}^3$ | -0.15966 |  |  |  || $\text{L}_0$ | $10^{-12} \text{ H}$ |  | 2.0765 |  |  || $\text{L}_1$ | $10^{-24} \text{ H/Hz}$ |  | -108.54 |  |  || $\text{L}_2$ | $10^{-33} \text{ H/Hz}^2$ |  | 2.1705 |  |  || $\text{L}_3$ | $10^{-42} \text{ H/Hz}^3$ |  | -0.01 |  |  || 阻抗 | Ω |  |  | 50 |  || 偏移延迟 | ps | 29.243 | 31.785 | 0 | 0 || 偏移损耗 | $\text{G}\Omega$ / s | 2.2 | 2.36 | 2.3 | 2.3 || 偏移 $Z_0$ | Ω | 50 | 50 | 50 | 50 || 参考 $Z_0$ | Ω | 50 | 50 | 50 | 50 |.. 注意：虽然 Keysight 为负载和直通校准标准指定了偏移损耗，但它们的偏移延迟为 0，实际上消除了传输线，因此应忽略这些损耗。偏移直通标准旨在作为未知的或齐平的直通标准。.. 重要：Keysight 85033E 的插头和插座校准标准具有几乎相同的参数，但有一个区别：开路标准（插座）的偏移损耗为 2.3 GΩ/s，而开路标准（插头）的偏移损耗为 2.2 GΩ/s。#### 创建要使用上述代码片段创建此校准套件：

In [ ]:
freq = skrf.Frequency(1, 9000, 1001, "MHz")
keysight_85033e_plug_open_std = keysight_calkit_open(
    freq,
    offset_delay=29.243e-12, offset_loss=2.2e9,
    c0=49.433e-15, c1=-310.13e-27, c2=23.168e-36, c3=-0.15966e-45,
    offset_z0=50, ref_z0=50
)
keysight_85033e_plug_short_std = keysight_calkit_short(
    freq,
    offset_delay=31.785e-12, offset_loss=2.36e9,
    l0=2.0765e-12, l1=-108.54e-24, l2=2.1705e-33, l3=-0.01e-42,
    offset_z0=50, ref_z0=50
)
load_std = keysight_calkit_load(freq)
thru_std = keysight_calkit_thru(freq)

#### 绘图##### 开路

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(12, 5)

plt.suptitle("Keysight 85033E Plug Open (S11)")
keysight_85033e_plug_open_std.plot_s_db(ax=ax[0], color='red', label="Magnitude")
keysight_85033e_plug_open_std.plot_s_deg_unwrap(ax=ax[1], color='blue', label="Phase")

##### 短路

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(12, 5)

plt.suptitle("Keysight 85033E Plug Short (S11)")
keysight_85033e_plug_short_std.plot_s_db(ax=ax[0], color='red', label="Magnitude")
keysight_85033e_plug_short_std.plot_s_deg_unwrap(ax=ax[1], color='blue', label="Phase")

#### 结论再次强调，在使用带有偏移的校准标准时，输入正确的偏移延迟至关重要。

### 示例：通用 SMA 插头校准套件现在我们来看一些完全不同的内容。作者使用 NanoRFE VNA6000，以 Keysight 85033E 校准标准为参考，测量了通用 SMA 标准的典型参数，数据如下。数值拟合后的电路参数如下所示。| 参数          | 单位                | 齐平开路   | 齐平开路   | 直通开路   || ------------- | ------------------- | -------- | -------- | -------- || VNA 连接器    |                     | SMA 母头 | 3.5 mm 母头 | SMA 插头 || DUT 连接器    |                     | SMA 插头 | SMA 插头 | SMA 直通+插头 || $\text{C}_0$ | $10^{-15} \text{ F}$ | 13.670   | 28.065   | 13.670   || 偏移延迟      | ps                  | 0        | 0        | 47.08    || 偏移 $Z_0$  | $\Omega$            | 50       | 50       | 50       || 参考 $Z_0$  | $\Omega$            | 50       | 50       | 50       |通用的*短路*标准实际上是一个齐平短路，因此仅显示通用的*开路*标准。由于测量噪声，仅进行了线性拟合。以上所有结果均为初步结果，尚未经过严格的验证。#### 创建

In [ ]:
freq = skrf.Frequency(1, 9000, 1001, "MHz")
generic_sma_plug_on_sma_socket_open_std = keysight_calkit_open(
    freq,
    offset_delay=0, offset_loss=0,
    c0=13.670e-15, c1=0, c2=0, c3=0,
    offset_z0=50, ref_z0=50
)
generic_sma_plug_on_apc35_socket_open_std = keysight_calkit_open(
    freq,
    offset_delay=0, offset_loss=0,
    c0=28.065e-15, c1=0, c2=0, c3=0,
    offset_z0=50, ref_z0=50
)
generic_sma_plug_on_sma_socket_thru_open_std = keysight_calkit_open(
    freq,
    offset_delay=47.08e-12, offset_loss=0,
    c0=13.670e-15, c1=0, c2=0, c3=0,
    offset_z0=50, ref_z0=50
)

#### 绘图##### 齐平开路

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(12, 5)

ax[0].set_title("Generic SMA Open Plug Phase (S11)")
generic_sma_plug_on_sma_socket_open_std.plot_s_deg_unwrap(ax=ax[0], color='red', label="SMA Port")
generic_sma_plug_on_apc35_socket_open_std.plot_s_deg_unwrap(ax=ax[0], color='blue', label="3.5 mm Port")

ax[1].set_title("Thru + SMA Plug Open Phase (S11)")
generic_sma_plug_on_sma_socket_thru_open_std.plot_s_deg_unwrap(ax=ax[1], color='red', label="SMA Port")

#### 结论当使用通用的SMA校准标准来校准SMA接口时，与理想的开路标准相比，相位误差小于10度。这证明了在非关键测量中可以采用理想化的假设。相比之下，当将通用的SMA开路标准与通用的直通适配器串联以形成SMA接口时，这实际上将标准转换为一个*偏移*标准，其相位偏差超过300度。在校准SMA插头和同轴电缆时，理想化的假设会导致较大的测量误差。可以通过确定直通标准的*偏移延迟*来纠正此问题，这相当于对所有测量应用端口扩展。当在SMA和3.5毫米接口上测量相同的通用开路标准时，观察到明显的差异。作者认为，不太可能是存在真实的物理边缘电容差异。更有可能是由于电路寄生效应，例如连接器有效参考平面在*连接*状态和*未连接*状态之间的变化（校准仅对*连接*端口有效，而不是对未端接的端口有效）。[[2](SOLT%20Calibration%20Standards%20Creation.html#ref2)]连接器寄生效应，例如引脚深度[[16](SOLT%20Calibration%20Standards%20Creation.html#ref16)][[17](SOLT%20Calibration%20Standards%20Creation.html#ref17)]以及在交叉匹配的3.5毫米和SMA/2.92毫米连接器对中的阶跃电容，也可能起作用。[[8](SOLT%20Calibration%20Standards%20Creation.html#ref8)]这可能是为什么不再在射频测量中使用无引脚空心开路标准（尽管在APC-7和N型连接器中，它们在历史计量学中具有可接受的结果）。由于缺少中心导体，开路标准的电气特性不稳定且依赖于端口。.. tip::   HP 85032B/E N型标准是一个历史示例，它采用无引脚的开路标准。Kurt Poulsen (OZ7OU) 发现，市面上流通的大多数通用N型标准（插头）都是它的克隆版本[[23](SOLT%20Calibration%20Standards%20Creation.html#ref23)]，因此可以在非关键校准中使用HP的定义[[22](SOLT%20Calibration%20Standards%20Creation.html#ref22)]。   但是，由于缺少匹配的中心导体延伸件，这不适用于插座类型。

## Rohde & Schwarz / 安立信 系数格式在 Rohde & Schwarz 和安立信 VNA 上，使用一种略有不同的格式来定义系数。以下是一个示例，展示了 Maury Microwave 8050CK10，它是一个 3.5 毫米、DC 到 26.5 GHz 的校准套件，其格式由 Rohde & Schwarz 定义。[[19](SOLT%20Calibration%20Standards%20Creation.html#ref19)]| 参数 | 单位 | 开路 | 短路 | 负载 | 直通 ||---|---|---|---|---|---|| $\text{C}_0$ | $10^{-15} \text{ F}$ | 62.54 | | | || $\text{C}_1$ | $10^{-15} \text{ F/GHz}$ | 1284.0 | | | || $\text{C}_2$ | $10^{-15} \text{ F/GHz}^2$ | 107.6 | | | || $\text{C}_3$ | $10^{-15} \text{ F/GHz}^3$ | -1.886 | | | || $\text{L}_0$ | $10^{-12} \text{ H}$ | | 0 | | || $\text{L}_1$ | $10^{-12} \text{ H/GHz}$ | | 0 | | || $\text{L}_2$ | $10^{-12} \text{ H/GHz}^2$ | | 0 | | || $\text{L}_3$ | $10^{-12} \text{ H/GHz}^3$ | | 0 | | || 电阻 | $\Omega$ | | | 50 | || 偏移长度 | mm | 4.344 | 5.0017 | 0 | 17.375 || 偏移损耗 | $\text{dB / }\sqrt{\text{GHz}}$ | 0.0033 | 0.0038 | 0 | 0.0065 |### 建模偏移传输线如上所示，本质上是相同的电路模型，唯一的区别在于偏移传输线以不同的测量单位定义：偏移延迟定义为物理长度而不是时间延迟，偏移损耗定义为分贝。偏移 $Z_0$ 被定义为系统阻抗 $Z_\text{ref}$（50 Ω 或 75 Ω），因此未列出。.. 提示::请注意，$S_{11}$ 测量中信号沿两个方向传播，因此以分贝表示的 $S_{11}$ 损耗与所陈述的偏移损耗相比，数值上是其两倍。在进行简单的单位转换后，我们可以重用 Keysight 模型中的相同计算。[[15](SOLT%20Calibration%20Standards%20Creation.html#ref15)]$$\begin{aligned}Z_\text{off} &= Z_\text{ref} \\t_\text{delay} &= \frac{D \cdot \sqrt{\epsilon_r}}{c_0} \\A_\text{loss} &= \frac{L \cdot Z_0}{t_\text{delay} \cdot 20 \log_{10}{(\mathrm{e})}}\end{aligned}$$其中，$D$ 和 $L$ 是 R&S 模型中的偏移长度（米）和偏移损耗（$\text{dB / }\sqrt{\text{GHz}}$），$t_\text{delay}$ 和 $A_\text{loss}$ 是 Keysight 模型中的偏移延迟（秒）和偏移损耗（$\Omega$ / s），$\epsilon_r$ 是介电常数，根据定义，它是空气，因此 $\epsilon_r = 1$，并且 $c_0$ 是光速。项 $20 \log_{10}{(\mathrm{e})}$ 是从分贝到奈培的转换。

In [ ]:
def rs_to_keysight(rs_offset_length, rs_offset_loss, offset_z0=50):
    offset_delay = rs_offset_length / skrf.constants.c
    offset_loss = skrf.mathFunctions.db_2_np(rs_offset_loss * offset_z0 / offset_delay)
    return offset_delay, offset_loss

进行单位转换后，我们可以像定义 Keysight 风格的系数中的校准标准一样定义标准。

In [ ]:
z0_ref = 50

offset_delay, offset_loss = rs_to_keysight(4.344e-3, 0.0033)
rlcg_open = offset_rlcg(freq, offset_delay, offset_loss, z0_ref)
medium_open = DistributedCircuit(frequency=freq,
    R=rlcg_open[0], L=rlcg_open[1], C=rlcg_open[2], G=rlcg_open[3],
    z0_port=z0_ref
)
line_open = medium_open.line(
    d=1, unit='m'
)

offset_delay, offset_loss = rs_to_keysight(5.0017e-3, 0.0038)
rlcg_short = offset_rlcg(freq, offset_delay, offset_loss, z0_ref)
medium_short = DistributedCircuit(frequency=freq,
    R=rlcg_short[0], L=rlcg_short[1], C=rlcg_short[2], G=rlcg_short[3],
    z0_port=z0_ref
)
line_short = medium_short.line(
    d=1, unit='m'
)

offset_delay, offset_loss = rs_to_keysight(17.375e-3, 0.0065)
rlcg_thru = offset_rlcg(freq, offset_delay, offset_loss, z0_ref)
medium_thru = DistributedCircuit(frequency=freq,
    R=rlcg_thru[0], L=rlcg_thru[1], C=rlcg_thru[2], G=rlcg_thru[3],
    z0_port=z0_ref
)
line_thru = medium_thru.line(
    d=1, unit='m'
)

### 建模并联阻抗并联阻抗的定义与 Keysight 格式相同。但是，请注意电容和电感所使用的单位！在给定的表中，电容的单位为 $10^{-15} \text{ F}$、$10^{-15} \text{ F/GHz}$、$10^{-15} \text{ F/GHz}^2$ 和 $10^{-15} \text{ F/GHz}^3$。对于 Keysight 和 Anritsu VNA，电容的单位为 $10^{-15} \text{ F}$、$10^{-27} \text{ F/Hz}$、$10^{-36} \text{ F/Hz}^2$ 和 $10^{-45} \text{ F/Hz}^3$。电感的单位也存在相同的差异。在开始建模之前，务必仔细检查单位。要将单位从第一种格式转换为第二种格式，将 $x_1$、$x_2$ 和 $x_3$ 乘以 1000（不要更改常数项 $x_0$）。为了保持一致性，我们将在代码中使用第二种格式。由于短路校准标准的电感被忽略，因此仅对开路校准标准的电容进行建模，短路被建模为理想状态。

In [ ]:
capacitor_poly = np.poly1d([
    -0.001886 * 1000e-45,
     0.1076   * 1000e-36,
    -1.284    * 1000e-27,
    62.54     * 1e-15
])
capacitor_open = capacitor_poly(freq.f)
shunt_open = ideal_medium.shunt_capacitor(capacitor_open) ** ideal_medium.open()
# or: shunt_open = ideal_medium.capacitor(capacitor_open) ** ideal_medium.short()
# see the Keysight example for explanation.

shunt_short = ideal_medium.short()

### 完成最后，我们将这些模型组件连接在一起。

In [ ]:
open_std = line_open ** shunt_open
short_std = line_short ** shunt_short
load_std = ideal_medium.match()
thru_std = line_thru

### 绘图再次，让我们检查校准标准的性能。#### 开路

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(12, 5)

plt.suptitle("Maury Microwave 8050CK10 Open (S11)")
open_std.plot_s_db(ax=ax[0], color='red', label="Magnitude")
open_std.plot_s_deg_unwrap(ax=ax[1], color='blue', label="Phase")

#### 短路

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(12, 5)

plt.suptitle("Maury Microwave 8050CK10 Short (S11)")
short_std.plot_s_db(ax=ax[0], color='red', label="Magnitude")
short_std.plot_s_deg_unwrap(ax=ax[1], color='blue', label="Phase")

#### 直通

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(12, 5)

plt.suptitle("Maury Microwave 8050CK10 Thru (S21)")
thru_std.s21.plot_s_db(ax=ax[0], color='red', label="Magnitude")
thru_std.s21.plot_s_deg_unwrap(ax=ax[1], color='blue', label="Phase")

### 结论结果与 Keysight 校准标准和通用的直通适配器类似。对于直通标准，S21 图显示了为什么添加电延迟有时可以作为一种粗略但可用的校准方法（“端口扩展”），用于 VNA 测量。同样，损耗非常低，相位偏移是导致 *偏移* 校准标准具有非理想特性的原因。

### 代码片段为了方便起见，您可以重用以下代码片段，从 Rohde & Schwarz 和 Anritsu 格式的系数中生成校准标准网络。```pythonimport numpy as npimport skrffrom skrf.media import DefinedGammaZ0def keysight_calkit_offset_line(freq, offset_delay, offset_loss, offset_z0, ref_z0):    if offset_delay or offset_loss:        r = offset_loss * offset_delay * np.sqrt(freq.f / 1e9)        l = (offset_delay * offset_z0) + r / (2 * np.pi * freq.f)        c = offset_delay / offset_z0        g = 0        medium = DistributedCircuit(frequency=freq,            R=r, L=l, C=c, G=g,            z0_port=ref_z0        )        offset_line = medium.line(d=1, unit='m')        return medium, offset_line    else:        medium = DefinedGammaZ0(frequency=freq, z0=ref_z0)        line = medium.line(d=0)        return medium, linedef rs_calkit_offset_line(freq, rs_offset_length, rs_offset_loss, offset_z0, ref_z0):    offset_delay = rs_offset_length / skrf.constants.c    if offset_delay != 0:        offset_loss = skrf.mathFunctions.db_2_np(rs_offset_loss * offset_z0 / offset_delay)    else:        offset_loss = 0    return keysight_calkit_offset_line(freq, offset_delay, offset_loss, offset_z0, ref_z0)def rs_calkit_open(freq, offset_length, offset_loss, c0, c1, c2, c3, offset_z0, ref_z0):    # 电容是相对于系统参考阻抗 ref_z0 定义的，而不是有损传输线阻抗。在 scikit-rf 中，`shunt_capacitor()` 和 `medium.open()` 方法的返回值（正确地）参考了 z0_port，该值已设置为 ref_z0。    medium, line = rs_calkit_offset_line(freq, offset_length, offset_loss, offset_z0, ref_z0)    if c0 or c1 or c2 or c3:        poly = np.poly1d([c3, c2, c1, c0])        capacitance = medium.shunt_capacitor(poly(freq.f)) ** medium.open()    else:        capacitance = medium.open()    return line ** capacitancedef rs_calkit_short(freq, offset_length, offset_loss, l0, l1, l2, l3, offset_z0, ref_z0):    # 电感是相对于系统参考阻抗 ref_z0 定义的，而不是有损传输线阻抗。在 scikit-rf 中，`shunt_inductance()` 和 `medium.short()` 方法的返回值（正确地）参考了 z0_port，该值已设置为 ref_z0。    medium, line = rs_calkit_offset_line(freq, offset_length, offset_loss, offset_z0, ref_z0)    if l0 or l1 or l2 or l3:        poly = np.poly1d([l3, l2, l1, l0])        inductance = medium.inductor(poly(freq.f)) ** medium.short()    else:        inductance = medium.short()    return line ** inductancedef rs_calkit_load(freq, offset_length, offset_loss, offset_z0, ref_z0):    medium, line = rs_calkit_offset_line(freq, offset_length, offset_loss, ref_z0, ref_z0)    load = medium.match()    return line ** loaddef rs_calkit_thru(freq, offset_length, offset_loss, offset_z0, ref_z0):    medium, line = rs_calkit_offset_line(freq, offset_length, offset_loss, offset_z0, ref_z0)    thru = medium.thru()    return line ** thrufreq = skrf.Frequency(1, 9000, 1001, "MHz")open_std = rs_calkit_open(    freq,    offset_length=4.344e-3, offset_loss=0.0033,    offset_z0=50, ref_z0=50,    # 由于单位差异，R&S 数据手册中的 c1、c2 和 c3 的数值必须乘以 1000。对于 Anritsu，则不需要。请检查您的数据手册中的单位！    c0=62.54     * 1e-15,    c1=-1.284    * 1000e-27,    c2=0.1076    * 1000e-36,    c3=-0.001886 * 1000e-45,)short_std = rs_calkit_short(    freq,    offset_length=5.0017e-3, offset_loss=0.0038,    offset_z0=50, ref_z0=50,    l0=0, l1=0, l2=0, l3=0)load_std = rs_calkit_load(    freq,    offset_length=0, offset_loss=0,    offset_z0=50, ref_z0=50,)thru_std = rs_calkit_thru(    freq,    offset_length=17.375e-3, offset_loss=0.0065,    offset_z0=50, ref_z0=50,)```

## 附录：偏移传输线的 γ 和 Zc 的推导### RLCG 线参数的推导根据 Keysight [[4](SOLT%20Calibration%20Standards%20Creation.html#ref4)]，公式 2B.3，*偏移*传输线的参数是相对于物理同轴线定义的，使用以下关系：$$\begin{aligned}A_\text{loss} &= \dfrac{R_0 v}{\sqrt{\epsilon_r \frac{f}{\text{1 GHz}}}} \\t_\text{delay} &= \dfrac{l \sqrt{\epsilon_r}}{v} = \sqrt{L_0 C} \\Z_\text{off} &= \frac{\mu_0 v}{2 \pi \sqrt{\epsilon_r}} \ln(\dfrac{D}{d}) = \sqrt{\frac{L_0}{C}}\end{aligned}$$其中 $v = c_0$ [[13](SOLT%20Calibration%20Standards%20Creation.html#ref13)]是光速，$μ_0$ 是真空透磁率，$\epsilon_r$ 是介质的相对介电常数，$R_0$ 和 $L_0$ 是分布的标称电阻和理想导体电感，$C$ 是分布电容，$D$ 和 $d$ 是同轴线的外部和内部直径。通过代数运算：$$\begin{aligned}\left( R_0 \sqrt{\dfrac{f}{10^9}} \right) l &= A_\text{loss} t_\text{delay} = \dfrac{R_0 v}{\sqrt{\epsilon_r \frac{f}{10^9}}} \dfrac{l \sqrt{\epsilon_r}}{v} \\L_0 &= t_\text{delay}  Z_\text{off} = \sqrt{L_0 C} \sqrt{\frac{L_0}{C}}  \\C &= \dfrac{t_\text{delay}}{Z_\text{off}} = \dfrac{\sqrt{L_0 C}}{\sqrt{\frac{L_0}{C}}} \\\end{aligned}$$最后，应用 [[4](SOLT%20Calibration%20Standards%20Creation.html#ref4)]，公式 2B.2 中的定义：$$\begin{aligned}G &= 0 \\L &= L_0 + \dfrac{R}{\omega}\end{aligned}$$完整的 RLCG 线参数的推导如下：$$\begin{aligned}R(f) l &= \left(A_\text{loss} t_\text{delay}\right) \cdot \sqrt{\frac{f}{10^9}} \\L(f) &= L_0 + L_\text{cond} = t_\text{delay}  Z_\text{off} + \dfrac{R}{2\pi f} \\C &= \dfrac{t_\text{delay}}{Z_\text{off}} \\G &= 0 \\l &= 1\text{ (归一化)}\end{aligned}$$### Keysight 的低损耗近似可以使用 RLCG 参数的通用公式计算线常数。$$\begin{aligned}Z_c &= \sqrt{ \frac{R + j \omega L}{G + j \omega C}} \\\gamma &= \sqrt{(R + j \omega L) (G + j \omega C)}\end{aligned}$$这是 scikit-rf 的 `DistributedCircuit` 中常用的方法。但是，Keysight 的官方出版物[[4](SOLT%20Calibration%20Standards%20Creation.html#ref4)]和其他文献[[15](SOLT%20Calibration%20Standards%20Creation.html#ref15)][[20](SOLT%20Calibration%20Standards%20Creation.html#ref20)]使用了低损耗近似来计算线常数。我们展示了公式和代码，以便手动计算 γ 和 Zc，并使用相同的低损耗近似。如果绝对需要与最后一位数完全相同的数值，这可能很有用。将 γ 表达式中的项重新组合（参见 [[21](SOLT%20Calibration%20Standards%20Creation.html#ref21)]，公式 2.83）：$$\begin{aligned}\gamma &= \sqrt{(R + j \omega L) (G + j \omega C)} \\       &= j\omega \sqrt{LC} \sqrt{1 - j \left(\frac{R}{\omega L} + \dfrac{G}{\omega C} \right) - \frac{RG}{\omega^2 LC}}\end{aligned}$$当 $G=0$ 时：$$\begin{aligned}Z_c &= \sqrt{ \frac{R + j \omega L}{j \omega C}} \\\gamma &= j\omega \sqrt{LC} \sqrt{1 - j \left(\frac{R}{\omega L}\right)}\end{aligned}$$由于 $L = L_0 + \frac{R}{\omega}$：$$\begin{aligned}Z_c &= \sqrt{ \frac{R + j \omega \left( L_0 + \frac{R}{\omega} \right) }{j \omega C}} \\    &= \sqrt{\dfrac{L_0}{C}} \sqrt{1 + \dfrac{R}{\omega L_0} (1 - j)} \\\gamma &= j\omega \sqrt{\left( L_0 + \frac{R}{\omega} \right) C} \sqrt{1 - j \left(\frac{R}{\omega \left( L_0 + \frac{R}{\omega} \right)}\right)} \\       &= j\omega \sqrt{L_0 C} \sqrt{1 + \dfrac{R}{\omega L_0} (1 - j)}\end{aligned}$$应用一阶泰勒展开 $\sqrt{1 + x} = 1 + x/2$ [[20](SOLT%20Calibration%20Standards%20Creation.html#ref20)]：$$\begin{aligned}\gamma &= j \omega \sqrt{L_0 C} \left[1 + (1 - j) \dfrac{R}{2\omega L_0}\right] \\Z_c &= \sqrt{\dfrac{L_0}{C}} \left[1 + (1 - j) \dfrac{R}{2 \omega L_0}\right]\end{aligned}$$通过将 R、L0 和 C 代入这些方程，最终可以得到 [[4](SOLT%20Calibration%20Standards%20Creation.html#ref4)] 中的官方公式：$$\begin{aligned}\alpha l &= \dfrac{A_\text{loss} t_\text{delay}}{2 Z_\text{off}} \sqrt{\dfrac{f}{10^9}} \\\beta l &= \omega t_\text{delay} + \alpha l \\\gamma l &= \alpha l + j \beta l \\Z_c &= Z_\text{off} + (1 - 1j) \left(\frac{A_\text{loss}}{4 \pi f}\right) \sqrt{\frac{f}{10^9}}\end{aligned}$$### 实现以下代码实现了 Keysight 的方法来计算线常数。```pythondef offset_gamma_and_zc(offset_delay, offset_loss, offset_z0):    alpha_l = (offset_loss * offset_delay) / (2 * offset_z0)    alpha_l *= np.sqrt(freq.f / 1e9)    beta_l = 2 * np.pi * freq.f * offset_delay + alpha_l    gamma_l = alpha_l + 1j * beta_l    zc = (offset_z0) + (1 - 1j) * (offset_loss / (4 * np.pi * freq.f)) * np.sqrt(freq.f / 1e9)    return gamma_l, zc```### 代码片段将以下定义替换到上面的代码片段中的 `keysight_calkit_offset_line()` 中，以复制确切的 Keysight 数据：```pythondef keysight_calkit_offset_line(freq, offset_delay, offset_loss, offset_z0, ref_z0):    if offset_delay or offset_loss:        alpha_l = (offset_loss * offset_delay) / (2 * offset_z0)        alpha_l *= np.sqrt(freq.f / 1e9)        beta_l = 2 * np.pi * freq.f * offset_delay + alpha_l        zc = offset_z0 + (1 - 1j) * (offset_loss / (4 * np.pi * freq.f)) * np.sqrt(freq.f / 1e9)        gamma_l = alpha_l + beta_l * 1j        medium = DefinedGammaZ0(frequency=freq, z0=zc, gamma=gamma_l, z0_port=ref_z0)        offset_line = medium.line(d=1, unit='m')        return medium, offset_line    else:        medium = DefinedGammaZ0(frequency=freq, z0=ref_z0)        line = medium.line(d=0)        return medium, line```

## References<div id="ref1"></div>\[1] “[Agilent Technologies 85056A 2.4 mm precision calibration kit](https://perma.cc/9M6N-PDXJ),” Agilent Technologies, User’s and Service Guide 85056–90020, Jan. 2002.<div id="ref2"></div>\[2] N. M. Ridler, “[Connectors, air lines and RF impedance](https://doi.org/10.1049/ic:20050150),” in 14th IEE Microwave Measurements Training Course, 2005, p. 9/1-9/18. "*the test port connector’s mating mechanism affects the established measurement reference plane which limits accurate characterisation as a standard. \[...] problem can be overcome either by depressing the mating mechanism using a dielectric plug or attaching a length of line to the centre conductor, terminated in an abrupt truncation. The dielectric plug technique is used as a standard with numerous VNA calibration kits.*"<div id="ref3"></div>\[3] [IEEE standard for precision coaxial connectors (DC to 110 GHz)](https://doi.org/10.1109/IEEESTD.2007.4317507), IEEE 287-2007, 2007. "*The reference plane for the Type N connector is the junction surface of the outer conductors \[...\] Unlike many of the other pin and socket connectors, the junction surface of the inner conductor is offset from the reference plane \[...\] The offset specifications in this document for the inner conductor pin differ from other common standards documents. Those documents specify the offset of the inner conductor pin as 5.283 mm through 5.360 mm (0.208 in through 0.211 in). This standard specifies the offset as 5.258 mm through 5.360 mm (0.207 in through 0.211 in).*"<div id="ref4"></div>\[4] “[Specifying calibration standards and kits for Agilent vector network analyzers](https://web.archive.org/web/20230605182500/https://people.ece.ubc.ca/robertor/Links_files/Files/AN-1287-11.pdf),” Agilent Technologies, Application Note 1287–11, Aug. 2009. See Equation 36, 37 for the propagation constant formulas of the offset transmission line.<div id="ref5"></div>\[5] P. J. Pupalaikis, [S-parameters for Signal Integrity](https://doi.org/10.1017/9781108784863). Cambridge: Cambridge University Press, 2020. Page 481. Written by a former signal integrity expert at LeCroy, the author of this tutorial recommends everyone who simulates or measures RF/microwave devices to get a copy. Each concept is accomplished by both formulas and executable code, making it an invaluable reference in this field.<div id="ref6"></div>\[6] “[Agilent Technologies 85033D 3.5 mm calibration kit](https://web.archive.org/web/20240406061304/https://web.ece.ucsb.edu/Faculty/rodwell/Classes/ECE218a/documentation/85033D_cal_kit_info.pdf),” Agilent Technologies, User’s and Service Guide 85033–90027, July 2002. "*The offset opens have inner conductors that are supported by a strong, low-dielectric constant plastic to minimize compensation values.*"<div id="ref7"></div>\[7] R. B. Marks and D. F. Williams, “[A general waveguide circuit theory](https://doi.org/10.6028/jres.097.024),” Journal of research of the National Institute of Standards and Technology, vol. 97, no. 5, pp. 533–562, 1992. "*This potentially useful result suggests that a particular line may be used as a calibration standard for any network analyzer with identical results. However, the assumption that v and i are continuous, which led to the result, is not generally valid. The example of a 50 Ω, 2.4 mm coaxial standard used on 50 Ω, 3.5 mm coaxial test ports makes this clear, for the standard must reflect the traveling waves even though its characteristic impedance is appropriate for a reflectionless standard. In general, the quality of the approximation depends in detail on the nature of the waveguide interface.*"<div id="ref8"></div>\[8] R. D. Pollard, “[Compensation technique improves measurements for a range of mechanically compatible connectors](https://web.archive.org/web/20250807132508/https://file.elecfans.com/web1/M00/61/D5/o4YBAFuGjPGAAB2WABS0KM2eFts501.pdf),” Microwave Journal, vol. 37, no. 10, p. 91+, Oct. 1994.<div id="ref9"></div>\[9]: J. R. Juroshek, C. A. Hoer, and R. F. Kaiser, “[Calibrating network analyzers with imperfect test ports](https://doi.org/10.1109/19.31010),” IEEE Transactions on Instrumentation and Measurement, vol. 38, no. 4, pp. 898–901, Aug. 1989, doi: 10.1109/19.31010.<div id="ref10"></div>\[10]: P. I. Somlo, “[The discontinuity capacitance and the effective position of a shielded open circuit in a coaxial line](https://archive.org/details/somlo1967),” Proceedings of the Institution of Radio and Electrical Engineers Australia, vol. 28, no. 1, pp. 7–9, Jan. 1967, doi: 10.5281/zenodo.17015632, [alt link](https://zenodo.org/records/17015632).<div id="ref11"></div>\[11] N. M. Ridler and M. Salter, “[Measuring the capacitance coefficients of coaxial open-circuits with traceability to national standards](https://perma.cc/HR8U-933S),” Microwave Journal, vol. 49, pp. 138–154, Oct. 2006.<div id="ref12"></div>\[12] N. M. Ridler, J. C. Medley, A. J. B. Fuller, and M. Runham, “[Computer generated equivalent circuit models for coaxial-line offset open circuits](https://doi.org/10.1049/ip-a-3.1992.0039),” IEE Proceedings A (Science, Measurement and Technology), vol. 139, no. 5, pp. 229–231, 1992.<div id="ref13"></div>\[13] “[Specifying calibration standards for the HP 8510 network analyzer](https://web.archive.org/web/20230620204616/https://hpmemoryproject.org/an/pdf/pn8510-5.pdf),” Hewlett-Packard, Product Note 8510–5, Mar. 1986. *Note the absence of any inductance parameters in both the text and the calibration standard datasheet on the last page.*<div id="ref14"></div>\[14] “[Agilent Technologies 85052D 3.5 mm economy calibration kit](https://web.archive.org/web/20250809205839/https://www.testunlimited.com/pdf/Keysight_85052D.pdf),” Agilent Technologies, User’s and Service Guide 85052–90079, Aug. 2010.<div id="ref15"></div>\[15] M. Wollensack and J. Hoffmann, “[METAS VNA Tools - Math Reference V2.9.0.](https://web.archive.org/web/20250829054357/https://www.metas.ch/dam/metas/en/data/fachbereiche/hochfrequenz/vna-tools/vnatools_v2_9_0/vnatools_math_v2.9.0-e.pdf.download.pdf/vnatools_math_v2.9.0-e.pdf)” Federal Institute of Metrology (METAS), Aug. 08, 2025. See Page 26, 27 for formulas of the Keysight and R&S coefficients. The Keysight formulas are equivalent to[[4](SOLT%20Calibration%20Standards%20Creation.html#ref4)].<div id="ref16"></div>\[16] K. Wong and J. Hoffmann, “[Improving VNA measurement accuracy by including connector effects in the models of calibration standards](https://doi.org/10.1109/ARFTG-2.2013.6737334),” in 82nd ARFTG Microwave Measurement Conference, Nov. 2013, pp. 1–7.<div id="ref17"></div>\[17] J. Ruefenacht and J. Hoffmann, “[Coaxial connector effects](https://perma.cc/9YLP-83J8),” presented at the EURAMET RF&MW expert meeting, Torino, May 2011. doi: 10.13140/2.1.3666.7685.<div id="ref18"></div>\[18] P. I. Somlo, “[The computation of coaxial line step capacitances](https://doi.org/10.1109/TMTT.1967.1126368),” IEEE Transactions on Microwave Theory and Techniques, vol. 15, no. 1, pp. 48–53, Jan. 1967.<div id="ref19"></div>\[19] “[3.5 mm Coaxial Calibration Kit - DC to 26.5 GHz - Models: 8050CK10/11 - 8050CK20/21,](https://web.archive.org/web/20250906090633/https://maurymw.com/wp-content/uploads/8050-511.pdf)” Maury Microwave, User Guide 8050–511, 2015. Source of the Maury Microwave 8050CK10 example. The coefficients of this calibration kit are given in multiple formats (Anritsu, Keysight, and R&S). You can compare their differences.<div id="ref20"></div>\[20] R. Monsalve, “[Effect of loss on VNA calibration standards](https://web.archive.org/web/20201230221105/https://loco.lab.asu.edu/loco-memos/edges_reports/report_20130807.pdf),” SESE, Arizona State University, Aug. 2013.<div id="ref21"></div> \[21] D. M. Pozar, Microwave engineering, Fourth edition. Hoboken, NJ: John Wiley & Sons, Inc, 2012.<div id="ref22"></div> \[22] “[Keysight vector network analyzer calibration kit standards definitions](https://web.archive.org/web/20250828172137/https://www.keysight.com/us/en/assets/9922-01521/technical-specifications/Calibration-Kit-Definitions.pdf).” Keysight Technologies, 2024.<div id="ref23"></div> \[23] K. Poulsen, “[How to fabricate a N male and female calibration kit with arbitrary calibration data for VNWA 2 and VNWA3](https://web.archive.org/web/20250916235928/https://www.hamcom.dk/VNWA/How%20to%20fabricate%20a%20N%20Male%20and%20Female%20calibration%20kit.pdf).” Feb. 2014.<div id="ref24"></div> \[24] J. Hoffmann, M. Wollensack, J. Ruefenacht, and M. Zeier, “[Extended S-parameters for imperfect test ports](https://doi.org/10.1088/0026-1394/52/1/121),” Metrologia, vol. 52, no. 1, p. 121, Jan. 2015, doi: 10.1088/0026-1394/52/1/121.<div id="ref25"></div> \[25] J. Ruefenacht and J. Hoffmann, “[Coaxial connector effects](https://perma.cc/9YLP-83J8),” presented at the EURAMET RF&MW expert meeting, Torino, May 2011. doi: 10.13140/2.1.3666.7685.<div id="ref26"></div> \[26] K. Wong and J. Hoffmann, “[Improving VNA measurement accuracy by including connector effects in the models of calibration standards](https://doi.org/10.1109/ARFTG-2.2013.6737334),” in 82nd ARFTG Microwave Measurement Conference, Nov. 2013, pp. 1–7. doi: 10.1109/ARFTG-2.2013.6737334.